# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**ML Task Framing:** **Supervised Priority Scoring / Learning to Rank** (with binary outcome target `is_declining_label`).

**Why this task type:** In content operations, editorial teams do not process an unsorted bucket of binary classes; they act on a prioritized weekly queue of the top N highest-opportunity pages to refresh. Supervised priority scoring predicts a continuous score $P(\text{declining}) \times \text{Impact}$ for each page, allowing us to sort candidate content from highest to lowest opportunity. This directly aligns with the operational decision of allocating limited editorial hours to the highest-leverage pages.

In [1]:
# Code check: Inspect target label binary distribution and basic ranking frame
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("=== TARGET LABEL DISTRIBUTION ===")
print(f"Total Pages: {len(df):,}")
print(f"Positive Class (Declining = 1): {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean():.1%})")
print(f"Negative Class (Stable/Up/Other = 0): {(df['is_declining_label'] == 0).sum():,} ({(df['is_declining_label'] == 0).mean():.1%})")

=== TARGET LABEL DISTRIBUTION ===
Total Pages: 30,000
Positive Class (Declining = 1): 16,262 (54.2%)
Negative Class (Stable/Up/Other = 0): 13,738 (45.8%)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target Definition:**
* **Proxy Label (Starter Dataset):** `is_declining_label = (trend_direction == 'down')`. This target indicates whether organic search performance declined in the trailing 30-day window relative to the prior 30-day period.
* **Observed Outcome (Warehouse Release):** In the panel data, the true target is the observed percentage drop in organic impressions or clicks in window $T_{t+30}$ relative to baseline $T_t$.
* **Target Leakage Safeguard:** Because `is_declining_label` is derived from `trend_direction` and `trend_pct`, both `trend_direction` and `trend_pct` must be strictly excluded from the feature matrix $X$.

In [2]:
# Code check: Target leakage audit — verifying excluded target derivations
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

all_cols = set(df.columns)
target_cols = {'trend_direction', 'trend_pct', 'is_declining_label'}
id_cols = {'content_id', 'client_id'}
feature_cols = [c for c in df.columns if c not in target_cols and c not in id_cols]

print(f"Total Columns: {len(all_cols)}")
print(f"Target & Derivations (EXCLUDED): {target_cols}")
print(f"Identifiers (CONTEXT ONLY): {id_cols}")
print(f"Clean Features Count: {len(feature_cols)}")

Total Columns: 44
Target & Derivations (EXCLUDED): {'is_declining_label', 'trend_direction', 'trend_pct'}
Identifiers (CONTEXT ONLY): {'content_id', 'client_id'}
Clean Features Count: 40


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric:** **Precision@K (specifically Precision@50 and Precision@100)** and **ROC-AUC**.

**Defending Precision@K:**
* A content team can typically refresh 50 to 100 articles per period. Precision@50 measures what percentage of the top 50 pages recommended by our model are actual declining pages in need of refresh.
* **Base Rate Benchmark:** Randomly choosing pages yields a baseline Precision of **54.2%**.
* **Heuristic Rule Benchmark:** A simple age rule (`days_since_last_update > 180`) yields **47.1% Precision** (worse than random!).
* **Target Model Performance:** A successful ML model must achieve **>75.0% Precision@50** and **>0.75 ROC-AUC**.

In [3]:
# Code check: Define Precision@K function and evaluate base rate vs heuristic baseline
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

def precision_at_k(df, score_col, k=50):
    top_k = df.sort_values(by=score_col, ascending=False).head(k)
    return top_k['is_declining_label'].mean()

# Base rate (random choice)
base_rate = df['is_declining_label'].mean()

# Simple rule score: days_since_last_update
rule_age_p50 = precision_at_k(df.fillna({'days_since_last_update': 0}), 'days_since_last_update', k=50)

print(f"Base Rate (Random Choice Precision): {base_rate:.1%}")
print(f"Heuristic Rule (Age > 180) Precision@50: {rule_age_p50:.1%}")

Base Rate (Random Choice Precision): 54.2%
Heuristic Rule (Age > 180) Precision@50: 52.0%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** One row = **One unique content page (`content_id`)** for a specific client (`client_id`) over a 90-day observation window.

**Dataset Dimensions:** 30,000 unique rows $\times$ 44 columns.

In [4]:
# Code check: Display slice of unit of analysis dataframe
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Dataframe Shape: {df.shape}")
print(f"Unique content_id count: {df['content_id'].nunique():,}")
print(f"Unique client_id count: {df['client_id'].nunique():,}")

sample_cols = ['content_id', 'client_id', 'content_type', 'search_volume', 'avg_position', 'ctr', 'days_since_last_update', 'trend_direction']
df[sample_cols].head()

Dataframe Shape: (30000, 44)
Unique content_id count: 30,000
Unique client_id count: 32


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why Simple Rules Fail:**
1. **Static Heuristics Fail Empirically:** Testing a common intuition (`days_since_last_update > 180`) yields **47.1% precision**, which is *worse than random guessing (54.2%)*. Content age alone does not cause decline—evergreen content retains rankings while high-competition topics decay regardless of age.
2. **Multi-Dimensional Feature Interactions:** Content decay depends on non-linear combinations of search volume, CPC, CTR tier, position tier (`striking` vs `page_1`), and engagement metrics (`scroll_rate`, `engagement_rate`). An if-statement cannot balance these competing forces.
3. **ML Efficiency:** Supervised models learn non-linear decision boundaries that combine all 40+ signals into a unified opportunity score.

In [5]:
# Code check: Demonstrate failure of simple rules vs multi-feature opportunity
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Compare decline rates across age tiers
age_decline = df.groupby('freshness_tier', observed=False)['is_declining_label'].agg(['count', 'mean'])
print("=== DECLINE RATE BY FRESHNESS TIER ===")
print(age_decline)

# Compare decline rates across position tiers
pos_decline = df.groupby('position_tier', observed=False)['is_declining_label'].agg(['count', 'mean'])
print("\n=== DECLINE RATE BY POSITION TIER ===")
print(pos_decline)

=== DECLINE RATE BY FRESHNESS TIER ===
                count      mean
freshness_tier                 
0-30            20480  0.511377
181+              174  0.471264
31-90             175  0.588571
91-180           9171  0.611057

=== DECLINE RATE BY POSITION TIER ===
               count      mean
position_tier                 
deep            1319  0.344200
page_1         11814  0.569663
page_3_5        7242  0.561585
striking        7304  0.609529
top_3           2321  0.240844


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.